# ENG-209, Partie 2, Série 9 (Cours)

## Introductioon à classification par régression logistique

Nous abordons maintenant une nouvelle famille de modèles : la classification.
Pour introduire cette thématique, nous allons utiliser une méthode particulièrement simple et puissante : la régression logistique.

Elle s’appuie sur les outils que nous avons déjà étudiés en régression linéaire, mais les adapte pour modéliser une probabilité et décider entre deux classes. C’est donc une transition naturelle entre les modèles de régression et les modèles de classification.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from numpy.typing import NDArray

In [ ]:
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["font.size"]      = 12
plt.rcParams['text.usetex']    = False
np.random.seed(42)

Prenons un exemple fictif très simple.

In [ ]:
X: NDArray[np.float64] = np.array([[0.5,0.75,1,1.25,1.5,1.75,2,2.25,2.5,2.75,4,4.25,4.5,4.75,5,5.25,5.5]]).T
y: NDArray[np.float64] = np.array([[0,0,0,0,0,0,0,1,0,1,1,1,1,1,1,1,1]], dtype=np.float64).T

Normalisons les variables indépendantes (explicatives) $X$ et créons la matrice $\tilde{X}$.

In [ ]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()

X_norm: NDArray[np.float64] = sc.fit_transform(X)
X_tilde: NDArray[np.float64] = np.column_stack([np.ones(X.shape[0]), X_norm])

A quoi ressemblent les données?

In [ ]:
fig, ax = plt.subplots()

ax.scatter(X_norm, y)

ax.set_xlabel('$x_1$')
ax.set_ylabel('$y$')

# N'affichons que 0 et 1 sur l'axe y (variable catégorique!)
ax.get_yaxis().set_ticks([0,1])
ax.grid(True)

plt.show()

### Régression linéaire suivie dun seuillage

Essayons tout d'abord un modèle de régression linéaire qui minimise la fonction de coût quadratique classique:

$$
g_{sl}(w) = \frac{1}{2N} \|y - \tilde{X}w\|_2^2
$$

et visualisons le résultat.

In [ ]:
from sklearn.linear_model import LinearRegression

model: LinearRegression = LinearRegression()
model.fit(X_norm, y)

intercept: np.float64 = np.array(model.intercept_).item()
coef: np.float64 = model.coef_.item()

In [ ]:
y_pred: NDArray[np.float64] = model.predict(X_norm)
x_half: float = (0.5 - intercept)/coef

fig, ax = plt.subplots()

ax.scatter(X_norm, y)
ax.plot(X_norm, y_pred, color='red')

ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$y$")

ax.plot(x_half, 0.5, 'gx')

ax.get_yaxis().set_ticks([0,1])
#ax.grid(True)

plt.show()

#### Seuillage: $\text{step}(x - 0.5)$

Et prenons la décision que, pour un $\tilde{X}^i$ donné, $\hat{y}^i = 1$ si notre modèle linéaire $\tilde{X}^i w \geq 0.5$, et $\hat{y}^i = 0$ si $\tilde{X}^i w < 0.5$.

Ceci définit la fonction $\text{step}(x - 0.5)$.

Veuillez vérifier que le point d'ordonnée $\hat{y} = 0.5$ correspond à  une abscisse $x_1 = \frac{0.5 - w_0}{w_1}$ (croix verte ci-dessus)

Visualisons cette frontière de décision ("step function" ci-dessous)

In [ ]:
x_half = (0.5 - intercept)/coef

fig, ax = plt.subplots()

ax.scatter(X_norm, y)
ax.plot(X_norm, y_pred, 'r')
ax.plot(X_norm, y_pred, 'rx')

ax.set_xlabel("$x_1$")
ax.set_ylabel("$y$")

# Afficher la marche
ax.plot([X_norm.min(), x_half],[0, 0], 'g')
ax.plot([x_half, x_half],[0, 1], 'g')
ax.plot([x_half, X_norm.max()],[1, 1], 'g')

ax.get_yaxis().set_ticks([0,1])

plt.show()

Les cercles bleus à gauche dont l'abscisse est inférieure à $x_1 = \frac{0.5 - w_0}{w_1}$ seront prédits $\hat{y} = 0$. La prédiction de ceux à droite sera $\hat{y} = 1$

Visualisons ça autrement

In [ ]:
X_y_0 = np.array(X_norm)[y == 0]
X_y_1 = np.array(X_norm)[y == 1]
X_y_hat_0 = np.array(X_norm)[y_pred < 0.5]
X_y_hat_1 = np.array(X_norm)[y_pred >= 0.5]


fig, ax = plt.subplots()

ax.set_xlabel("$x_1$")

ax.scatter(X_y_0, np.zeros(len(X_y_0)), color='green', s=250, alpha=0.3, label="$y = 0$")
ax.scatter(X_y_1, np.zeros(len(X_y_1)), color='blue', s=250, alpha=0.3, label="$y = 1$")

ax.plot(X_y_hat_0, np.zeros(len(X_y_hat_0)), 'bo', label=r"$\hat{y} = 0$")
ax.plot(X_y_hat_1, np.zeros(len(X_y_hat_1)), 'ro', label=r"$\hat{y} = 1$")

ax.get_yaxis().set_ticks([])

ax.legend(loc='best')

plt.show()

#### Types d'erreur de prédiction de classification

Nous commettons une erreur de classification lorsque $y$ et $\hat{y}$ diffèrent.

Si $y = 1$ mais que $\hat{y} = 0$, l'erreur est un **faux négatif** (*false negative*)

Si $y = 0$ mais que $\hat{y} = 1$, l'erreur est un **faux positif** (*false positive*)

Il y a également naturellement les vrais positifs ($y = \hat{y} = 1)$ et les vrais négatifs ($y = \hat{y} = 0)$

Dans notre cas, veuillez vérifier qu'il y a **7** vrais positifs, **8** vrais négatifs, **0** faux positif et **2** faux négatifs, pour un total de N=17 observations. Ces valeurs sont nécessaires pour établir la **matrice de confusion** (*confusion matrix*).

### La sigmoïde logistique

Bien que simple, la méthode précédente est loin d'être idéale.

Il serait opportun de non pas modéliser la variable qualitative $y$ mais la probabilité que celle-ci se réalise.

Nous essayons donc de modéliser explicitement $p := ℙ\left( \hat{y} = 1 \mid X \right)$. Cette probabilité sera toujours comprise entre 0 et 1, et donc toute tentative pour ajuster un nuage de probabilité par une droite sera invalidée par le fait que la droite n'est pas bornée (comme dans l'exemple précédent). La transformation de $p$ en $\frac{p}{1-p}$ permet de travailler sur des valeurs variant de 0 à $+ \infty$, puis le passage au logarithme permet de travailler sur un nuage de points dont les valeurs varient entre $- \infty$ et $+ \infty$, ce qui rend possible l'approximation par une droite réelle. Sous cette forme on peut tenter un ajustement du nuage de points.

Nous allons donc modéliser $\ln \left( \frac{p}{1-p} \right)$ par une droite. C'est-à-dire:

$$\ln \left( \frac{p}{1-p} \right) = \tilde{X} w$$

La probabilité $p$ peut donc s'écrire (à vérifier):

$$p := ℙ\left(\hat{y} = 1 | X\right) = \frac{e^{\tilde{X}w}}{1 + e^{\tilde{X}w}} = \frac{1}{1 + e^{-\tilde{X}w}} = \sigma\left({\tilde{X}w}\right)$$

... avec, $\sigma(x) = \frac{1}{1 + e^{-x}}$, la fonction de **sigmoïde logistique**.

In [ ]:
sigmoid_function = lambda x: 1.0/(1.0 + np.exp(-x))

A quoi ressemble cette fonction, la sigmoïde logistique?

In [ ]:
a = np.arange(-20,20,0.2)

fig, ax = plt.subplots()

ax.plot(a, sigmoid_function(10*a), label=r'$\sigma(10*a)$')
ax.plot(a, sigmoid_function(a), label=r'$\sigma(a)$')
ax.plot(a, sigmoid_function(0.4*a), label=r'$\sigma(0.6*a)$')
ax.plot(a, sigmoid_function(.2*a), label=r'$\sigma(0.2*a)$')

ax.set_xlabel(r'$a$')
ax.legend(loc='best')

plt.show()

Et récrivons la fonction de coût quadratique:

$$g_\text{sl}(w) = \frac{1}{2N} \sum_{i=1}^N \left( \sigma\left(\tilde{X}^i w\right) - y^i \right)^2$$

In [ ]:
# Least squares cost for logistic regression
g_sl = lambda w: 1/(2*y.shape[0]) * np.sum( (sigmoid_function(X_tilde.dot(w)) - y)**2 )

#### Vous pouvez vérifier que le gradient de la fonction $g_\text{lsc}(w)$ est:
    
$$\nabla g_{sl}(w) = \frac{1}{N} \sum_{i=1}^{N} \left( \sigma \left(\tilde{X}^i w \right) - y^i \right) \sigma\left(\tilde{X}^i w\right) \left( 1-\sigma \left(\tilde{X}^i w \right) \right) \tilde{X}^i$$

La somme des gradients en chaque point i de 1 à N s'écrit dès lors:

In [ ]:
def gradient_sl_forloop(w):
    my_sum = []
    for i in range(X_tilde.shape[0]):
        sigma_ = sigmoid_function(X_tilde[i].dot(w))[0]
        my_sum.append((sigma_-y[i]).dot(sigma_)*(1-sigma_)*X_tilde[i,:])
    return (1/y.shape[0])*np.sum(my_sum,0).reshape(2,1)

La notation vectorielle suivante donne le même résultat mais de façon plus efficace

In [ ]:
def gradient_sl(w):
    sigma_ = sigmoid_function(X_tilde.dot(w))
    f = (sigma_ - y).dot(sigma_.T)
    p = np.diag((1-sigma_).reshape(X_tilde.shape[0],))
    
    return (1/y.shape[0])*np.diagonal(f.dot(p)).T.dot(X_tilde).reshape(2,1)

#### Descente de gradient

In [ ]:
w      = np.array([[-4],[-4]], dtype=np.float64)
alpha  = 10

maxits = 5000

In [ ]:
# On garde l'historique pour affichage
w_0_history = []
w_1_history = []
g_history   = []

for i in range(maxits):
    w_0_history.append(w[0][0])
    w_1_history.append(w[1][0])
    g_history.append(g_sl(w))
    
    w = w - alpha*gradient_sl(w)

In [ ]:
print(f"w_0 = {w[0][0]:2.2f}, w_1 = {w[1][0]:2.2f}")

#### Examinons cette surface de coût $\left( (w_0, w_1), g_{lsc}\right)$

In [ ]:
# Préparons les variables pour afficher cette surface
w_0: NDArray[np.float64] = np.arange(-7, 7, 0.05, dtype=np.float64)
w_1: NDArray[np.float64] = np.arange(-7, 7, 0.05, dtype=np.float64)

w_grid: NDArray[np.float64] = np.array(np.meshgrid(w_0, w_1)).T.reshape(-1,2)

g_sl_grid: NDArray[np.float64] = np.array([g_sl(w_it.reshape(2,1)) for w_it in w_grid.reshape(-1,2)])

cost_matrix: NDArray[np.float64] = np.array(g_sl_grid).reshape(w_0.shape[0], w_1.shape[0])

In [ ]:
import plotly.graph_objects as go

figx = go.Figure(
    data = [
        go.Surface(x=w_0, y=w_1, z=cost_matrix.T, opacity=0.7,
            contours={"z": dict(
                show=True,
                usecolormap=True,
                highlightcolor="limegreen",
                project_z=True,
            )
            }
        ),
        go.Scatter3d(x=w_0_history, y=w_1_history, z=g_history, mode ='markers', marker_size=4, marker_color='blue')
    ]
)

figx.update_layout(scene = dict(
                    xaxis_title='w_0',
                    yaxis_title='w_1',
                    zaxis_title='Coûts quadratiques'),
                    width=1600, height=800)

figx.show(renderer="browser")
#figx.show(renderer='notebook')

In [ ]:
y_pred_sl = sigmoid_function(X_tilde.dot(w))

fig, ax = plt.subplots()

ax.scatter(X_norm, y)


ax.plot(X_norm, y_pred_sl, 'g')
ax.plot(X_norm[y_pred_sl>=0.5], y_pred_sl[y_pred_sl>=0.5], 'ro', label=r"$\hat{y} = 1$")
ax.plot(X_norm[y_pred_sl<0.5], y_pred_sl[y_pred_sl<0.5], 'rx', label=r"$\hat{y} = 0$")

ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$y$")

ax.get_yaxis().set_ticks([0,1])

ax.legend(loc='best')

plt.show()

Cette fonction de coût est en général très "plate" à plusieurs endroits, ce qui peut rendre sa minimisation lente et compliquée.

Elle n'est d'ailleurs pas idéalement adaptée à une classification de type binaire ($y = \{0,1\}$)

### Minimisation de l'entropie croisée 

**Maximum de vraisemblance (loi de Bernoulli)**:

Nous rappelons qu’une variable aléatoire $Y$ suit une loi de Bernoulli si $P(Y=y\ |\ X) = p^y\ (1-p)^{1-y},\quad y=\{0,\ 1\}$.

On souhaite maximiser la vraisemblance sur $N$ observations:

$$L(p) = {\displaystyle \prod_{i=1}^{N}}\ ({p^i})^{y^i}\ (1-{p^i})^{1-{y^i}}$$

En prenant le logarithme pour transformer un produit en somme et en se rappelant que $p^i = P\left(\hat{y}^i = 1 | \tilde{X}^i\right) = \sigma\left({\tilde{X^i}w}\right)$, la fonction de coût (à minimiser) s'écrit:

$$g_\text{cel} = -\frac{1}{N} \sum_{i=1}^{N} \left( y^i \ln\left(\sigma\left(\tilde{X}^i w\right)\right) + (1-y^i) \ln\left(1-\sigma\left(\tilde{X}^i w\right)\right)\right)$$

Veuillez noter le signe (-), qui transforme un problème de maximisation en un problème de minimisation.

In [ ]:
g_cel = lambda w: -1.0/y.shape[0] * np.sum( (y.T.dot(np.log(sigmoid_function(X_tilde.dot(w)))) + (1-y).T.dot(np.log(1-sigmoid_function(X_tilde.dot(w)))) ) )

Le gradient s'écrit (à vérifier):

$$\nabla g_{ll}(w) = \frac{2}{N} \tilde{X}^T \left(\sigma\left(\tilde{X}w\right) - y\right)$$

In [1]:
gradient_cel = lambda w: 2.0/y.shape[0] * X_tilde.T.dot(sigmoid_function(X_tilde.dot(w)) - y)

#### Descente de gradient

In [ ]:
w = np.array([[-4],[-4]], dtype=np.float64)
alpha = 1

maxits = 5000

In [ ]:
# On garde l'historique pour affichage
w_0_history = []
w_1_history = []
g_history   = []

for i in range(maxits):
    w_0_history.append(w[0][0])
    w_1_history.append(w[1][0])
    g_history.append(g_cel(w))
    
    w = w - alpha*gradient_cel(w)

In [ ]:
print(f"w_0 = {w[0][0]:2.2f}, w_1 = {w[1][0]:2.2f}")

In [ ]:
# Préparons les variables pour afficher cette surface

w_0 = np.arange(-9, 9, 0.05, dtype=np.float64)
w_1 = np.arange(-9, 9, 0.05, dtype=np.float64)

w_grid = np.array(np.meshgrid(w_0, w_1)).T.reshape(-1,2)

g_cel_grid = np.array([g_cel(w_it.reshape(2,1)) for w_it in w_grid.reshape(-1,2)])

cost_matrix = np.array(g_cel_grid).reshape(w_0.shape[0], w_1.shape[0])

In [ ]:
import plotly.graph_objects as go

figx = go.Figure(
    data = [
        go.Surface(x=w_0, y=w_1, z=cost_matrix.T, opacity=0.8,
            contours={"z": dict(
                show=True,
                usecolormap=True,
                highlightcolor="limegreen",
                project_z=True,
            )
            }),
            go.Scatter3d(x=w_0_history, y=w_1_history, z=g_history, mode ='markers', marker_size=4, marker_color='blue')
        ]
    )

figx.update_layout(scene = dict(
                    xaxis_title='w_0',
                    yaxis_title='w_1',
                    zaxis_title='Coûts entropie croisée'),
                    width=1600, height=800)

figx.show(renderer="browser")
#figx.show(renderer='notebook')

In [ ]:
y_pred_cel = sigmoid_function(X_tilde.dot(w))

fig, ax = plt.subplots()

ax.scatter(X_norm, y, label=r"$(x,y)$")
ax.scatter(X_norm[y_pred_cel>=0.5], y_pred_cel[y_pred_cel>=0.5], marker='o', color='red', label=r"$\hat{y} = 1$")
ax.scatter(X_norm[y_pred_cel<0.5], y_pred_cel[y_pred_cel<0.5], marker='x', color='red', label=r"$\hat{y} = 0$")
ax.plot(X_norm, y_pred_sl, color='green', linestyle=':', label='Quadratique')
ax.plot(X_norm, y_pred_cel, color='blue', label='Entropie croisée')

ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$y$")

ax.get_yaxis().set_ticks([0,1])

ax.legend()

plt.show()

## La Régression Logistique avec Scikit-Learn

In [ ]:
from sklearn.linear_model import LogisticRegression

# Régularisation: 'none' ou 'l2'
# Solver: lbfgs
clf = LogisticRegression(solver='lbfgs' ,penalty=None) 
clf.fit(X_norm, y.reshape(y.shape[0],))

In [ ]:
w_opt_0 = clf.intercept_.item()
w_opt_1 = clf.coef_.item()

In [ ]:
print("w_0: ", w_opt_0, ", w_1: ", w_opt_1)

In [ ]:
x_half = - w_opt_0 / w_opt_1

# Logistic function
test_X = np.arange(min(X_norm).item(), max(X_norm).item(), 0.05)
test_y = sigmoid_function(w_opt_0 + test_X.dot(w_opt_1))
test_Y = sigmoid_function(w[0][0] + test_X.dot(w[1][0]))

fig, ax = plt.subplots()

ax.scatter(X_norm, y)
ax.plot(x_half, 0.5, 'bx')




ax.plot(test_X, test_y, 'r')
ax.plot(test_X, test_Y, 'b--')

ax.set_xlabel(r"$x_1$")
ax.set_ylabel(r"$y$")

plt.show()

**Matrice de confusion**: Evaluons la performance de notre *classifier*

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_predict: NDArray[np.bool] = clf.predict(X_norm)

cm: NDArray[np.int64] = confusion_matrix(y, y_predict)

fig, ax = plt.subplots(figsize=(9,6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clf.classes_)

disp.plot(ax=ax)

ax.set_xlabel("Étiquettes prédites")
ax.set_ylabel("Étiquettes réelles")

plt.show()

In [ ]:
y_predict_proba: NDArray[np.float64] = clf.predict_proba(X_norm)

In [ ]:
fig, ax = plt.subplots()

ax.plot(y_predict_proba[:,1], 'o-r', label=r"$P(y=1 | X)$")
ax.plot([0, 16],[0.08, 0.08],':', label=r"Threshold (0.08)")
ax.legend(loc='best')

plt.show()

In [ ]:
y_predict = (y_predict_proba[:,1] >= 0.08).astype(bool)

In [ ]:


cm = confusion_matrix(y, y_predict)

fig, ax = plt.subplots(figsize=(9,6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clf.classes_)

disp.plot(ax=ax)

ax.set_xlabel("Étiquettes prédites")
ax.set_ylabel("Étiquettes réelles")

## Prenons maintenant le jeu de données classique "iris"

Ce dataset contient 4 variables indépendantes et la classe / le label {0: setosa, 1: versicolor, 2: virginica}

In [ ]:
import pandas as pd
from sklearn import datasets

X_iris_, y_iris_ = datasets.load_iris(return_X_y=True,as_frame=True)

## Renomons les colonnes
X_iris_.columns=['sepal_long', 'sepal_larg', 'petal_long', 'petal_larg']

y_iris_ds: pd.Series = pd.Series(y_iris_, dtype=int)

# Ne gardons que longueur et largeur du pétale, et ignorons la classe 0: setosa qui a peu d'intérêt
X_iris_df: pd.DataFrame = X_iris_[["petal_long", "petal_larg"]][y_iris_ds > 0]
y_iris_ds = y_iris_ds[y_iris_ds > 0]

X_iris: NDArray[np.float64] = X_iris_df.to_numpy()
y_iris: NDArray[np.int8] = y_iris_ds.to_numpy()

In [ ]:
import plotly.graph_objects as go

figx = go.Figure(data = [
    go.Scatter3d(x=X_iris_df.petal_long, y=X_iris_df.petal_larg, z=y_iris_ds,
                        mode ='markers', marker_size=4)
                       ])

figx.update_scenes(xaxis_title_text='Petal length', yaxis_title_text='Petal width', zaxis_title_text='Class')
figx.update_layout(autosize=False, width=800, height=600)

figx.show(renderer='browser')
#figx.show(renderer='notebook')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
 
#scaler = StandardScaler()
#X[num_cols] = scaler.fit_transform(X[num_cols])

X_iris_train, X_iris_test, y_iris_train, y_iris_test = train_test_split(X_iris_df, y_iris_ds, test_size=0.4, stratify=y_iris_ds)

model_iris_cls = LogisticRegression(penalty=None)

model_iris_cls.fit(X_iris_train, y_iris_train)

Evaluation de la performance (matrice de confusion)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_iris_predict = model_iris_cls.predict(X_iris_test)
cm = confusion_matrix(y_iris_test, y_iris_predict)

fig, ax = plt.subplots(figsize=(9,6))

disp = ConfusionMatrixDisplay(confusion_matrix=cm)

disp.plot(ax=ax)

ax.set_xlabel("Étiquettes prédites")
ax.set_ylabel("Étiquettes réelles")

plt.show()

Paramètres du modèle

In [ ]:
w_0 = model_iris_cls.intercept_.item()
w_1 = model_iris_cls.coef_[0][0]
w_2 = model_iris_cls.coef_[0][1]

In [ ]:
# 1 rangée de 3 sous-graphes disposés horizontalement
fig, ax = plt.subplots(1, 3, figsize=(20, 6))

k: NDArray[np.float64] = np.arange(3.5, 6.5, 0.01, dtype=np.float64).reshape(-1,1)

# --- TRAIN ---
ax[0].plot(X_iris_train[y_iris_train==1].petal_long, X_iris_train[y_iris_train==1].petal_larg, 'bo', label=r"$y_{train}=0$")
ax[0].plot(X_iris_train[y_iris_train==2].petal_long, X_iris_train[y_iris_train==2].petal_larg, 'rx', label=r"$y_{train}=1$")
ax[0].plot(k, -(w_0+w_1*k)/w_2, 'g--')
ax[0].axis([3.5,6.8,1.0,2.6])
ax[0].legend(loc='upper left')
ax[0].set_title("Train")

# --- TEST ---
ax[1].plot(X_iris_test[y_iris_test==1].petal_long, X_iris_test[y_iris_test==1].petal_larg, 'bo', label=r"$y_{test}=0$")
ax[1].plot(X_iris_test[y_iris_test==2].petal_long, X_iris_test[y_iris_test==2].petal_larg, 'rx', label=r"$y_{test}=1$")
ax[1].plot(k, -(w_0+w_1*k)/w_2, 'g--')
ax[1].axis([3.5,6.8,1.0,2.6])
ax[1].legend(loc='upper left')
ax[1].set_title("Test")

# --- PREDICTIONS ---
ax[2].plot(X_iris_test[y_iris_predict==1].petal_long, X_iris_test[y_iris_predict==1].petal_larg, 'bo', label=r"$\hat{y}_{test}=0$")
ax[2].plot(X_iris_test[y_iris_predict==2].petal_long, X_iris_test[y_iris_predict==2].petal_larg, 'rx', label=r"$\hat{y}_{test}=1$")
ax[2].plot(k, -(w_0+w_1*k)/w_2, 'g--')
ax[2].axis([3.5,6.8,1.0,2.6])
ax[2].legend(loc='upper left')
ax[2].set_title("Predictions")

plt.tight_layout()
plt.show()